# Nassau Candy Distributor — Phase 4: KPI Metrics (Pandas)

This notebook rebuilds the core KPIs from the Phase 3 SQL analysis directly in pandas. This is necessary because the Streamlit app (Phase 5) will be deployed to Streamlit Community Cloud, which cannot connect to a local MySQL database — so all KPI logic needs a pandas/CSV-based equivalent.

Each KPI below has been cross-checked against its corresponding SQL query and confirmed to produce identical results. Final outputs are exported as CSV files for Streamlit to load directly.

In [26]:
import pandas as pd
import numpy as np
df = pd.read_csv(r"C:\Users\kehka\Desktop\Unified Mentor\Nassau Candy Project\nassau_candy_cleaned.csv")

print(df.shape)
df.head()

(9994, 24)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Country/Region,City,State/Province,Postal Code,...,Sales,Units,Gross Profit,Cost,Lead Time,Year Gap,Normalized Lead Time,Factory,Route_State,Route_Region
0,1,US-2021-103800-CHO-MIL-31000,2024-01-03,2026-06-30,Standard Class,103800,United States,Houston,Texas,77095,...,6.50,2,4.22,2.28,909,2,179,Wicked Choccy's,Wicked Choccy's -> Texas,Wicked Choccy's -> Interior
1,2,US-2021-112326-CHO-TRI-54000,2024-01-04,2026-07-01,Standard Class,112326,United States,Naperville,Illinois,60540,...,7.50,2,4.90,2.60,909,2,179,Wicked Choccy's,Wicked Choccy's -> Illinois,Wicked Choccy's -> Interior
2,3,US-2021-112326-CHO-NUT-13000,2024-01-04,2026-07-01,Standard Class,112326,United States,Naperville,Illinois,60540,...,10.47,3,7.47,3.00,909,2,179,Lot's O' Nuts,Lot's O' Nuts -> Illinois,Lot's O' Nuts -> Interior
3,4,US-2021-112326-CHO-SCR-58000,2024-01-04,2026-07-01,Standard Class,112326,United States,Naperville,Illinois,60540,...,10.80,3,7.50,3.30,909,2,179,Lot's O' Nuts,Lot's O' Nuts -> Illinois,Lot's O' Nuts -> Interior
4,5,US-2021-141817-CHO-TRI-54000,2024-01-05,2026-07-05,Standard Class,141817,United States,Philadelphia,Pennsylvania,19143,...,11.25,3,7.35,3.90,912,2,182,Wicked Choccy's,Wicked Choccy's -> Pennsylvania,Wicked Choccy's -> Atlantic


In [27]:
## Confirm the columns
df.columns.tolist() 

['Row ID',
 'Order ID',
 'Order Date',
 'Ship Date',
 'Ship Mode',
 'Customer ID',
 'Country/Region',
 'City',
 'State/Province',
 'Postal Code',
 'Division',
 'Region',
 'Product ID',
 'Product Name',
 'Sales',
 'Units',
 'Gross Profit',
 'Cost',
 'Lead Time',
 'Year Gap',
 'Normalized Lead Time',
 'Factory',
 'Route_State',
 'Route_Region']

## Delay Threshold

Lead Time ≥ 1634 marks the boundary of Cluster 3, the slowest of three distinct sub-groups identified in the Phase 2 Lead Time investigation (histogram + Year Gap analysis). This threshold is evidence-based, not arbitrary, and is used consistently with the SQL analysis (Query 24).

In [28]:
## Set the delay threshold
delay_threshold = 1634

## Phase 4A: Overall Delay Frequency

Delay Frequency is calculated at the **order level** (an order counts as delayed if any of its line items meets the threshold), matching the revised SQL Query 24/24b. Cross-checked against SQL: 8,389 total orders, 2,790 delayed, 33.26% — confirmed identical.

In [29]:
## Phase 4A: Calculate overall delay frequency

total_orders = df["Order ID"].nunique()

delayed_orders = df.loc[
    df["Lead Time"] >= delay_threshold,
    "Order ID"
].nunique()

overall_delay_frequency = (
    delayed_orders / total_orders * 100
)

print("Total Orders:", total_orders)
print("Delayed Orders:", delayed_orders)
print("Overall Delay Frequency:", round(overall_delay_frequency, 2), "%")

Total Orders: 8389
Delayed Orders: 2790
Overall Delay Frequency: 33.26 %


## Route-Level Delay Frequency

Same order-level logic, broken out by `Route_State`. Filtered to routes with at least 10 orders, matching the volume floor used throughout the SQL route analysis.

In [30]:
## Route-level delay frequency

route_delay = (
    df.groupby("Route_State")
      .agg(
          Route_Volume=("Order ID", "nunique"),
          Delayed_Orders=("Order ID", 
                          lambda x: x[df.loc[x.index, "Lead Time"] >= delay_threshold].nunique())
      )
      .reset_index()
)

route_delay["Delay_Frequency_Pct"] = (
    route_delay["Delayed_Orders"] /
    route_delay["Route_Volume"] * 100
)

route_delay["Delay_Frequency_Pct"] = (
    route_delay["Delay_Frequency_Pct"].round(2)
)

route_delay = route_delay[
    route_delay["Route_Volume"] >= 10
].sort_values(
    "Delay_Frequency_Pct",
    ascending=False
)

route_delay.head(10)

,Route_State,Route_Volume,Delayed_Orders,Delay_Frequency_Pct
29,Lot's O' Nuts -> New Mexico,18,11,61.11
158,Wicked Choccy's -> New Mexico,14,8,57.14
84,Secret Factory -> Washington,11,6,54.55
58,Secret Factory -> Illinois,13,7,53.85
118,The Other Factory -> New York,13,7,53.85
5,Lot's O' Nuts -> Connecticut,40,21,52.50
13,Lot's O' Nuts -> Iowa,14,7,50.00
142,Wicked Choccy's -> Iowa,12,6,50.00
166,Wicked Choccy's -> Rhode Island,18,9,50.00
72,Secret Factory -> New York,23,11,47.83


## Phase 4B: Route Efficiency Score

Normalizes each qualifying route's average Lead Time onto a 0–100 scale (100 = fastest, 0 = slowest), matching SQL Query 25. This is the formal implementation of the relative-ranking approach adopted after the Lead Time investigation — no absolute day-count claims, only relative comparison.

In [31]:
## Phase 4B: Route Efficiency Score

route_stats = (
    df.groupby("Route_State")
      .agg(
          Route_Volume=("Order ID", "nunique"),
          Avg_Lead_Time=("Lead Time", "mean")
      )
      .reset_index()
)

route_stats = route_stats[
    route_stats["Route_Volume"] >= 10
].copy()

route_stats.head()

,Route_State,Route_Volume,Avg_Lead_Time
0,Lot's O' Nuts -> Alabama,30,1295.205882
1,Lot's O' Nuts -> Arizona,94,1306.198198
2,Lot's O' Nuts -> Arkansas,25,1285.354839
3,Lot's O' Nuts -> California,944,1316.910222
4,Lot's O' Nuts -> Colorado,82,1347.533981


In [32]:
## Calculate the Efficiency Score

max_lead_time = route_stats["Avg_Lead_Time"].max()
min_lead_time = route_stats["Avg_Lead_Time"].min()

route_stats["Efficiency_Score"] = (
    100 *
    (max_lead_time - route_stats["Avg_Lead_Time"]) /
    (max_lead_time - min_lead_time)
).round(2)

route_efficiency = route_stats.sort_values(
    "Efficiency_Score",
    ascending=False
).reset_index(drop=True)

route_efficiency.head(10)

,Route_State,Route_Volume,Avg_Lead_Time,Efficiency_Score
0,Wicked Choccy's -> Nevada,10,1182.250000,100.00
1,Secret Factory -> Texas,16,1227.062500,85.39
2,Lot's O' Nuts -> Virginia,91,1229.486239,84.60
3,Wicked Choccy's -> South Carolina,14,1234.210526,83.05
4,Lot's O' Nuts -> Utah,21,1236.000000,82.47
5,Wicked Choccy's -> Virginia,85,1238.877358,81.53
6,Wicked Choccy's -> Mississippi,17,1240.454545,81.02
7,Lot's O' Nuts -> Nevada,23,1243.250000,80.11
8,Lot's O' Nuts -> Oregon,62,1245.194805,79.47
9,Lot's O' Nuts -> Kansas,15,1248.866667,78.27


## Phase 4C: Top 10 and Bottom 10 Routes

Fastest and slowest routes by average Lead Time, matching SQL Queries 10 and 11. Cross-checked and confirmed identical, both routes and scores.

In [33]:
# Phase 4C: Top and Bottom Routes

top_routes = (
    route_stats
    .sort_values(
        ["Avg_Lead_Time", "Route_Volume"],
        ascending=[True, False]
    )
    .head(10)
    .reset_index(drop=True)
)

bottom_routes = (
    route_stats
    .sort_values(
        ["Avg_Lead_Time", "Route_Volume"],
        ascending=[False, False]
    )
    .head(10)
    .reset_index(drop=True)
)

print("Top 10 Fastest Routes")
display(top_routes)

print("\nBottom 10 Slowest Routes")
display(bottom_routes)

Top 10 Fastest Routes


,Route_State,Route_Volume,Avg_Lead_Time,Efficiency_Score
0,Wicked Choccy's -> Nevada,10,1182.250000,100.00
1,Secret Factory -> Texas,16,1227.062500,85.39
2,Lot's O' Nuts -> Virginia,91,1229.486239,84.60
3,Wicked Choccy's -> South Carolina,14,1234.210526,83.05
4,Lot's O' Nuts -> Utah,21,1236.000000,82.47
5,Wicked Choccy's -> Virginia,85,1238.877358,81.53
6,Wicked Choccy's -> Mississippi,17,1240.454545,81.02
7,Lot's O' Nuts -> Nevada,23,1243.250000,80.11
8,Lot's O' Nuts -> Oregon,62,1245.194805,79.47
9,Lot's O' Nuts -> Kansas,15,1248.866667,78.27



Bottom 10 Slowest Routes


,Route_State,Route_Volume,Avg_Lead_Time,Efficiency_Score
0,Wicked Choccy's -> New Mexico,14,1488.882353,0.00
1,Lot's O' Nuts -> Iowa,14,1479.000000,3.22
2,Secret Factory -> Washington,11,1472.363636,5.39
3,Lot's O' Nuts -> New Mexico,18,1456.944444,10.42
4,Lot's O' Nuts -> Connecticut,40,1420.553191,22.28
5,Wicked Choccy's -> Iowa,12,1403.785714,27.75
6,Wicked Choccy's -> Indiana,44,1402.462963,28.18
7,Wicked Choccy's -> Tennessee,56,1393.761194,31.02
8,Lot's O' Nuts -> Tennessee,88,1390.963303,31.93
9,Secret Factory -> New York,23,1379.625000,35.63


## Phase 4D: High-Volume + Poor-Performance Routes

Flags routes with both above-average shipment volume AND above-average Lead Time — the routes that matter most operationally, not just statistically. Matches SQL Query 26 exactly (same 6 routes, same order).

In [34]:
## Phase 4D — High-Volume + Poor-Performance Routes.
## Step 4D.1 — Calculate the thresholds

avg_route_volume = route_stats["Route_Volume"].mean()
avg_route_lead_time = route_stats["Avg_Lead_Time"].mean()

print("Average Route Volume:", round(avg_route_volume, 2))
print("Average Route Lead Time:", round(avg_route_lead_time, 2))

Average Route Volume: 91.46
Average Route Lead Time: 1321.25


In [35]:
## Step 4D.2 — Identify the bottleneck routes

high_volume_poor_performance = route_stats[
    (route_stats["Route_Volume"] > avg_route_volume) &
    (route_stats["Avg_Lead_Time"] > avg_route_lead_time)
].copy()

high_volume_poor_performance = (
    high_volume_poor_performance
    .sort_values(
        ["Route_Volume", "Efficiency_Score"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

high_volume_poor_performance

,Route_State,Route_Volume,Avg_Lead_Time,Efficiency_Score
0,Lot's O' Nuts -> New York,544,1330.516279,51.65
1,Lot's O' Nuts -> Washington,230,1343.962825,47.26
2,Lot's O' Nuts -> Ohio,218,1334.114504,50.47
3,Wicked Choccy's -> Pennsylvania,188,1339.220264,48.81
4,Wicked Choccy's -> Washington,187,1375.723982,36.90
5,Wicked Choccy's -> Illinois,183,1329.128571,52.10


## Ship Mode Cost-Time Tradeoff

Compares cost and Lead Time across Ship Mode categories, matching SQL Query 27. Notably, Ship Mode does not show the expected real-world relationship with Lead Time (e.g., Same Day averaging higher Lead Time than Standard Class) — further evidence that the Order Date/Ship Date relationship does not reflect genuine shipping chronology.

In [36]:
## Ship mode stats
ship_mode_stats = (
    df.groupby("Ship Mode")
      .agg(
          Total_Orders=("Order ID", "nunique"),
          Total_Cost=("Cost", "sum"),
          Avg_Cost_Per_Shipment=("Cost", "mean"),
          Avg_Lead_Time=("Lead Time", "mean"),
          Lead_Time_Variability=("Lead Time", "std"),
          Total_Gross_Profit=("Gross Profit", "sum")
      )
      .reset_index()
)

# Round for readability
ship_mode_stats[["Total_Cost", "Avg_Cost_Per_Shipment", "Avg_Lead_Time", 
                  "Lead_Time_Variability", "Total_Gross_Profit"]] = ship_mode_stats[
    ["Total_Cost", "Avg_Cost_Per_Shipment", "Avg_Lead_Time", 
     "Lead_Time_Variability", "Total_Gross_Profit"]
].round(2)

ship_mode_stats = ship_mode_stats.sort_values("Avg_Lead_Time").reset_index(drop=True)

ship_mode_stats

,Ship Mode,Total_Orders,Total_Cost,Avg_Cost_Per_Shipment,Avg_Lead_Time,Lead_Time_Variability,Total_Gross_Profit
0,Standard Class,5026,28240.27,4.73,1315.06,261.49,54895.06
1,Second Class,1621,9410.92,4.84,1324.37,261.99,18014.38
2,Same Day,440,2397.94,4.42,1333.91,252.75,4663.38
3,First Class,1302,7272.98,4.73,1338.23,265.58,13935.41


## Export KPI Results

Saving each KPI result as a separate CSV for the Streamlit dashboard to load directly, avoiding the need to recompute this logic inside the app itself.

In [37]:
## Exporting CSV
route_efficiency.to_csv("route_efficiency.csv", index=False)
route_delay.to_csv("route_delay_frequency.csv", index=False)
top_routes.to_csv("top_10_routes.csv", index=False)
bottom_routes.to_csv("bottom_10_routes.csv", index=False)
high_volume_poor_performance.to_csv("high_volume_poor_performance.csv", index=False)
ship_mode_stats.to_csv("ship_mode_performance.csv", index=False)

print("All KPI files exported successfully.")

All KPI files exported successfully.


## Phase 4 Complete — Summary

All KPIs from the Problem Statement have been rebuilt in pandas and cross-verified against the Phase 3 SQL results:

- Overall Delay Frequency: 33.26% (8,389 orders, 2,790 delayed)
- Route Efficiency Score: normalized 0–100 scale, all qualifying routes scored
- Top 10 / Bottom 10 Routes: identified and verified
- High-Volume + Poor-Performance Routes: 6 routes flagged (Washington appears twice, across both major factories)
- Ship Mode Cost-Time Tradeoff: no meaningful cost differentiation across modes; Lead Time differences run counter to real-world expectations

**Exported files:** `route_efficiency.csv`, `route_delay_frequency.csv`, `top_10_routes.csv`, `bottom_10_routes.csv`, `high_volume_poor_performance.csv`, `ship_mode_performance.csv`

**Next phase:** Streamlit dashboard (5 pages: Overview, Route Efficiency, Geographic Map, Ship Mode Comparison, Route Drill-Down).